In [1]:
import pandas as pd
import numpy as np

# import cleaned csv file

file_path = '..\\DataCleaning\\cleaned_dataset.csv'

try:
    df = pd.read_csv(file_path)
    print("File loaded successfully.")
except FileNotFoundError:
    print("File not found in the specified path.")
except PermissionError:
    print("Permission denied to read the file.")
except Exception as e:
    print(f"An error occurred: {e}")

File loaded successfully.


Handling text and categorical variables (Feature Engineering)

In [2]:
categorical_variables = df.select_dtypes(include=['object']).columns
categorical_variables

Index(['Name', 'Gender', 'Location', 'Email', 'Phone', 'Address', 'Segment',
       'Timestamp', 'ProductList', 'Plan', 'Start_Date', 'End_Date',
       'TotalInteractionType', 'FirstInteractionDate', 'LastInteractionDate',
       'most_recent_action_date', 'Frequency', 'Comment', 'customer_segment'],
      dtype='object')

In [3]:
df['SubscriptionPlan'].value_counts()

KeyError: 'SubscriptionPlan'

In [ ]:
# Ordinal Encoding for categorical variables
from sklearn.preprocessing import OrdinalEncoder

ordinal_encoder = OrdinalEncoder(categories=[['Basic', 'Student', 'Essential', 'Unlimited', 'Bronze', 'Plus', 'Silver', 'Select', 'Gold', 'Prime', 'Eco', 'Flex', 'Deluxe', 'Smart', 'Pro', 'VIP', 'Express', 'Family', 'Trial', 'Elite']])
df['SubscriptionPlan_encoded'] = ordinal_encoder.fit_transform(df[['SubscriptionPlan']])

To avoid introducing a large number of columms into the dataset with One-Hot Encoding of the SubscriptionPlan column, I decided to use Ordinal Encoding instead.

In [ ]:
# One-hot Encoding for Gender variable

gender_dummies = pd.get_dummies(df['Gender'], prefix='Gender')
df = pd.concat([df, gender_dummies], axis=1)


In [ ]:
# One-hot Encoding for Segment column
segment_dummies = pd.get_dummies(df['Segment'], prefix='Segment')
df = pd.concat([df, segment_dummies], axis=1)

In [ ]:
# One-hot Encoding for FrequencyOfInteractions column
frequency_dummies = pd.get_dummies(df['FrequencyOfInteractions'], prefix='FrequencyOfInteractions')
df = pd.concat([df, frequency_dummies], axis=1)

In [ ]:
# Optimise the data for memory efficiency
# Convert float columns to float32 and int columns to int32 for memory optimization
# Select only numeric columns for validation
numeric_cols = df.select_dtypes(include=[np.number])
print(numeric_cols)

for col in numeric_cols.columns:
    if df[col].dtype == "float64":
        df[col] = df[col].astype("float32")
    elif df[col].dtype == "int64":
        df[col] = df[col].astype("int32")

# Display final dataset summary
print("Final Dataset Info After Optimization:")
df.info()

In [ ]:
df.columns

In [ ]:
# Drop irrelevant columns
df = df.drop(['CustomerID', 'Gender', 'Segment', 'FrequencyOfInteractions', 'SubscriptionPlan'], axis=1)


In [ ]:
y = df['ChurnLabel']
X = df.drop(['ChurnLabel'], axis=1)

In [ ]:
# Perform the test-train split, setting apart 20%
from sklearn.model_selection import train_test_split

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Model Training using RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

In [ ]:
predictions = model.predict(X_test)
predictions

In [ ]:
# Evaluate accuracy
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")

The accuracy is 97.20% meaning that the model has captured the pattern in the data. I will therefore proceed to fine-tune the model to see whether it will improve the accuracy

Model tuning
 - RandomizedSearchCV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

parameter_grid = {
    "n_estimators": [10, 50, 100], 
    "max_features": [2, 8, 13],
    "max_depth": [2, 10, None],
}

# Use RandomizedSearchCV instead of GridSearchCV
random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=parameter_grid,
    n_iter=10, 
    cv=3,  
    scoring="accuracy",  # Use accuracy for classification
    return_train_score=True,
    n_jobs=-1,  # Use all available CPU cores for parallel processing
    random_state=42
)

# Fit the model
random_search.fit(X, y)

In [ ]:
print(f"Best Parameters RandomizedSearchCV: {random_search.best_params_}")

In [ ]:
final_model = RandomForestClassifier(n_estimators=100, max_features=13, max_depth=2, random_state=42)
final_model.fit(X_train, y_train)

In [ ]:
predictions_encoded = final_model.predict(X_test)
predictions

In [ ]:
accuracy = accuracy_score(y_test, predictions_encoded)
print(f"Accuracy: {accuracy:.4f}")

for i in range(5):  # Show first 5 examples
    print(f"Actual: {([y_test.iloc[i]])[0]}, Predicted: {predictions[i]}")

The final accuracy is 97.24% which is a slight improvement by 0.04%. This model is accurate enough for production and I will demonstrate the script to make it deployable

In [ ]:
import joblib
joblib.dump(final_model, "customer_churn_prediction_model.pk1")

The model can now be deployed in production.